# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EimanZahra1472/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/EimanZahra1472/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 142 (delta 50), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.89 MiB | 18.40 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/flyrank-ml-internship-starter
(30000, 45)


In [2]:
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)

bucket1 = df.groupby("stale")["is_declining"].agg(["mean", "count"])
bucket1.columns = ["decline_rate", "n"]
print(bucket1)
print(f"\nOverall decline rate (base rate): {df['is_declining'].mean():.3f}")

       decline_rate      n
stale                     
0          0.542480  29826
1          0.471264    174

Overall decline rate (base rate): 0.542


In [3]:
df["position_tier"] = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 1000],
                               labels=["1-3", "4-10", "11-20", "21+"])

tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["below_tier_avg_ctr"] = (df["ctr"] < tier_avg_ctr).astype(int)

bucket2 = df.groupby("below_tier_avg_ctr")["is_declining"].agg(["mean", "count"])
bucket2.columns = ["decline_rate", "n"]
print(bucket2)

                    decline_rate      n
below_tier_avg_ctr                     
0                       0.409098   6265
1                       0.577165  23735


/tmp/ipykernel_3435/1514955908.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")


Signal 1 (staleness, behind FlyRank's refresh flags): Bucketed by stale = days_since_last_update >= 180. Decline rate for non-stale pages: 54.2% (n=29,826). Decline rate for stale pages: 47.1% (n=174). Verdict: OPPOSITE. Stale pages in this dataset are slightly less likely to be declining than fresh ones, and the stale group is tiny (0.6% of inventory)  this matches what I found in Week 1/2, where almost no pages met a stale+visible threshold at all. Staleness alone isn't a useful decline signal here.

Signal 2 (CTR vs. position-tier average, behind FlyRank's CTR-fix flag): Bucketed by whether a page's CTR falls below its own position tier's average CTR. Decline rate for below-tier-average pages: 57.7% (n=23,735). Decline rate for at/above-tier-average pages: 40.9% (n=6,265). Verdict: CONFIRMED. A ~17-point gap on solid sample sizes both sides  underperforming CTR relative to peers at the same position is a real, usable signal for decline.

The rule, in plain words: A page is worth reviewing first if it is underperforming its expected CTR for its position and it's getting meaningful traffic  staleness turned out not to help, so it's dropped from the rule. Volume (impressions) is added as a gate so the rule doesn't chase low-traffic noise.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# --- Build the transparent baseline score ---
# Rule (plain words): a page is worth reviewing first if its CTR underperforms
# its own position tier's average AND it has meaningful traffic (impressions).
# Staleness was tested and dropped — it didn't separate decliners from non-decliners.

VOLUME_THRESHOLD = 500  # matches the "visible" threshold used in notebooks 01/02

df["visible"] = (df["impressions_90d"] >= VOLUME_THRESHOLD).astype(int)

# Score: readable on purpose — no fitted weights, just simple conditions
df["baseline_action_score"] = (
    df["below_tier_avg_ctr"] * df["visible"] * df["impressions_90d"]
)

# Reason code — ONE code, as the card asks
df["reason_code"] = "ctr_underperforms_position_tier"

# Action label
df["action"] = np.where(
    df["baseline_action_score"] > 0,
    "review_for_ctr_fix",
    "no_action"
)

# Rank
ranked = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)

# --- Precision@K, with base rate for context ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining"].values
base_rate = y.mean()

for k in (20, 50):
    p_k = precision_at_k(df["baseline_action_score"], y, k)
    print(f"Precision@{k}: {p_k:.3f}   (base rate: {base_rate:.3f})")

# --- Write the CSV (not committed to git — regenerates on every run) ---
import os
os.makedirs("work/outputs", exist_ok=True)

output_cols = ["content_id", "client_id", "baseline_action_score", "reason_code",
               "action", "impressions_90d", "avg_position", "ctr", "is_declining"]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"\nWrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
ranked[output_cols].head(10)


Precision@20: 0.500   (base rate: 0.542)
Precision@50: 0.400   (base rate: 0.542)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_action_score,reason_code,action,impressions_90d,avg_position,ctr,is_declining
0,content_5fe46e04994d,client_4e07408562,517715,ctr_underperforms_position_tier,review_for_ctr_fix,517715,4.2,0.14,1
1,content_aaef01a50def,client_19581e27de,517109,ctr_underperforms_position_tier,review_for_ctr_fix,517109,5.4,0.25,0
2,content_8c19996aa890,client_4e07408562,509252,ctr_underperforms_position_tier,review_for_ctr_fix,509252,2.5,0.15,1
3,content_2cb567c3c89b,client_6208ef0f77,497727,ctr_underperforms_position_tier,review_for_ctr_fix,497727,22.2,0.10,0
4,content_4c36c775b818,client_4e07408562,463103,ctr_underperforms_position_tier,review_for_ctr_fix,463103,2.3,0.41,1
5,content_2dba2b1f9536,client_6208ef0f77,443434,ctr_underperforms_position_tier,review_for_ctr_fix,443434,27.9,0.21,0
6,content_1a9e894be2e2,client_19581e27de,416180,ctr_underperforms_position_tier,review_for_ctr_fix,416180,4.0,0.23,1
7,content_2c2606c5d176,client_19581e27de,347399,ctr_underperforms_position_tier,review_for_ctr_fix,347399,4.2,0.53,1
8,content_db5989a78dd3,client_4e07408562,345111,ctr_underperforms_position_tier,review_for_ctr_fix,345111,5.4,0.21,0
9,content_44e481c8f55b,client_19581e27de,312694,ctr_underperforms_position_tier,review_for_ctr_fix,312694,1.4,0.65,0


The rule, in code: baseline_action_score = below_tier_avg_ctr × visible × impressions_90d, using only same-window observable signals. Reason code: ctr_underperforms_position_tier. Action: review_for_ctr_fix when score > 0, else no_action.

Result: Precision@20 = 0.500, Precision@50 = 0.400  both below the base rate of 0.542. This is an honest negative: multiplying by raw impressions_90d means the rule effectively just ranks by page size among CTR-underperforming pages, and page size alone isn't predictive of decline. The CONFIRMED signal check in section 1 was a real average-level pattern (below-tier-CTR pages decline more often on average), but that doesn't survive being turned into a ranking dominated by volume. This baseline is weak by design a fair, beatable floor for the modeling week, not a result to be proud of.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top20 = ranked.head(20)
for i, row in top20.iterrows():
    correct = "✓ correct" if row["is_declining"] == 1 else "✗ WRONG"
    print(f"{i+1}. {row['content_id']} | action: {row['action']} | reason: {row['reason_code']}")
    print(f"   impressions={row['impressions_90d']}, position={row['avg_position']}, ctr={row['ctr']} | actually declining={row['is_declining']} ({correct})")
    print()



1. content_5fe46e04994d | action: review_for_ctr_fix | reason: ctr_underperforms_position_tier
   impressions=517715, position=4.2, ctr=0.14 | actually declining=1 (✓ correct)

2. content_aaef01a50def | action: review_for_ctr_fix | reason: ctr_underperforms_position_tier
   impressions=517109, position=5.4, ctr=0.25 | actually declining=0 (✗ WRONG)

3. content_8c19996aa890 | action: review_for_ctr_fix | reason: ctr_underperforms_position_tier
   impressions=509252, position=2.5, ctr=0.15 | actually declining=1 (✓ correct)

4. content_2cb567c3c89b | action: review_for_ctr_fix | reason: ctr_underperforms_position_tier
   impressions=497727, position=22.2, ctr=0.1 | actually declining=0 (✗ WRONG)

5. content_4c36c775b818 | action: review_for_ctr_fix | reason: ctr_underperforms_position_tier
   impressions=463103, position=2.3, ctr=0.41 | actually declining=1 (✓ correct)

6. content_2dba2b1f9536 | action: review_for_ctr_fix | reason: ctr_underperforms_position_tier
   impressions=443434, p

1.content_5fe46e04994d — flagged, pos 4.2, CTR 0.14. Declining ✓. High volume + weak CTR at a strong position — a believable review case.

2.content_aaef01a50def — flagged, pos 5.4, CTR 0.25. Stable ✗. Would be wrong if this CTR is simply normal for this page's content type or intent, not a symptom of decline.

3.content_8c19996aa890 — flagged, pos 2.5, CTR 0.15. Declining ✓. Strong position with low CTR is a solid signal.

4.content_2cb567c3c89b — flagged, pos 22.2, CTR 0.10. Stable ✗. Position 22 is already a weak tier — "below tier average" means little when the tier itself converts poorly.
5.content_4c36c775b818 — flagged, pos 2.3, CTR 0.41. Declining ✓. Notably high CTR (0.41) to still be "below tier average" — suggests this tier's average is unusually high at this position.

6.content_2dba2b1f9536 — flagged, pos 27.9, CTR 0.21. Stable ✗. Same low-tier issue as row 4.

7.content_1a9e894be2e2 — flagged, pos 4.0, CTR 0.23. Declining ✓.

8.content_2c2606c5d176 — flagged, pos 4.2, CTR 0.53. Declining ✓, despite a relatively high CTR — decline here may be driven by something the score doesn't capture (e.g. dropping impressions, not CTR).

9.content_db5989a78dd3 — flagged, pos 5.4, CTR 0.21. Stable ✗.

10.content_44e481c8f55b — flagged, pos 1.4, CTR 0.65. Stable ✗. Clearest miss so far — CTR 0.65 at position 1.4 is strong by any normal standard; only flagged because its tier average is even higher.

11.content_cb112fce36be — flagged, pos 5.6, CTR 0.16. Declining ✓.

12.content_9532f197bbc8 — flagged, pos 2.0, CTR 0.87. Declining ✓, despite a very high CTR (0.87) — another case where "below tier average" doesn't mean "objectively bad." Worth flagging that at very top positions, tier averages may be skewed by a few outlier high-CTR pages, making even strong performers look like underperformers.

13.content_36ff89c8214e — flagged, pos 7.3, CTR 0.05. Stable ✗. Genuinely low CTR but stable anyway — would be wrong if this page's low CTR is a stable characteristic (e.g. low-intent query) rather than a symptom of change.

14.content_b28d1efd668f — flagged, pos 26.2, CTR 0.06. Stable ✗. Weak tier again — same failure mode as rows 4, 6.

15.content_8451fc6f034d — flagged, pos 2.3, CTR 0.03. Stable ✗. This one's harder to explain away — very strong position (2.3) with very low CTR (0.03) and still stable. Worth a manual look; may indicate a title/meta mismatch that's chronic rather than declining.

16.content_aa4baf490b43 — flagged, pos 5.9, CTR 0.5. Stable ✗. High CTR (0.5) — likely another tier-average skew case.

17.content_008fb02c46cb — flagged, pos 4.4, CTR 0.26. Declining ✓.

18.content_813e88069237 — flagged, pos 26.2, CTR 0.06. Declining ✓ — weak tier, but this time correctly flagged, showing the low-tier cases aren't uniformly wrong, just noisier.

19.content_ff94c9b6b411 — flagged, pos 27.4, CTR 0.04. Declining ✓.

20.content_c84a0ab98e90 — flagged, pos 7.8, CTR 0.03. Stable ✗

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
weak_picks = top20[top20["is_declining"] == 0]
print(f"{len(weak_picks)} of the top 20 are false positives")
weak_picks[["content_id", "avg_position", "ctr", "impressions_90d"]]

score_inputs = ["below_tier_avg_ctr", "visible", "impressions_90d"]
label_derived_cols = ["trend_direction", "trend_pct", "is_declining"]
leak_found = any(col in score_inputs for col in label_derived_cols)
print(f"\nLabel-derived columns in score inputs: {leak_found}")
print(f"Score built from: {score_inputs} — same-window observable signals only.")


10 of the top 20 are false positives

Label-derived columns in score inputs: False
Score built from: ['below_tier_avg_ctr', 'visible', 'impressions_90d'] — same-window observable signals only.


Weak picks: 10 of the top 20 are false positives  flagged for review but not actually declining (rows 2, 4, 6, 9, 10, 13, 14, 15, 16, 20). Two clear failure patterns: (1) weak-tier positions (rows 4, 6, 14  positions 22–28) where "below tier average" is a low bar since the whole tier underperforms anyway, and (2) high-CTR pages still counted as "below average" (rows 10, 12, 16  CTR 0.5–0.87) because their tier average is skewed even higher, likely by a handful of outlier top performers. Row 15 (position 2.3, CTR 0.03, stable) is the hardest to explain  a genuinely weak CTR at a strong position that hasn't led to decline, worth a manual look rather than dismissal.

Leakage check: confirmed clean. Label-derived columns in score inputs: False. The score is built only from below_tier_avg_ctr, visible, and impressions_90d  all same-window observable signals, no trend_direction, trend_pct, or label-derived field used.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.